In [ ]:
!pip install umap-learn pandas matplotlib seaborn scikit-learn

In [ ]:
# ==========================================
# THESIS BLOCK 1: Aggregated Confusion Matrices by Prompt Family
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from scipy.special import softmax

# --- 1. Aesthetics & Setup ---
sns.set_context("talk")
sns.set_style("white") # Clean background for matrices

# --- 2. Load Data ---
vision_emb = np.load('cached_vision_embeddings.npy')
text_emb = np.load('cached_text_embeddings_custom.npy')
logit_scale = float(np.load('flair_logit_scale.npy'))
vision_meta = pd.read_csv('cached_vision_metadata.csv')
text_meta = pd.read_csv('cached_text_metadata_custom.csv')

# Preprocess vision metadata to ensure clean labels
valid_idx = vision_meta['adjudicated_dr_grade'].notna()
vision_emb = vision_emb[valid_idx]
vision_meta = vision_meta[valid_idx].reset_index(drop=True)
y_true = vision_meta['adjudicated_dr_grade'].astype(int).values

# --- 3. Mathematical Constraints & Preparation ---
# Constraint Step 1a: L2 Normalize Vision Embeddings
v_norm = vision_emb / np.linalg.norm(vision_emb, axis=1, keepdims=True)

families = ['T_label', 'T_expert', 'T_layman', 'T_exclusion']
grade_order = ['noDR', 'mildDR', 'modDR', 'sevDR', 'prolDR']

# --- 4. Plotting & Aggregation ---
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, fam in enumerate(families):
    # Initialize an empty 5x5 matrix to aggregate across the 10 variations
    agg_cm = np.zeros((5, 5), dtype=int)

    for var_num in range(1, 11):
        # Extract the 5 text embeddings for this specific variation (ordered noDR to prolDR)
        t_sub = np.zeros((5, 512))
        for g_idx, grade in enumerate(grade_order):
            # Find the exact row in text_meta matching family, variation, and grade
            mask = (text_meta['Family'] == fam) & (text_meta['Variation_Number'] == var_num) & (text_meta['Grade'] == grade)
            t_idx = text_meta.index[mask].tolist()[0]
            t_sub[g_idx] = text_emb[t_idx]

        # Constraint Step 1b: L2 Normalize Text Embeddings
        t_norm = t_sub / np.linalg.norm(t_sub, axis=1, keepdims=True)

        # Constraint Step 2 & 3: Cosine Similarity multiplied by the learned temperature (Logit Scale)
        logits = (v_norm @ t_norm.T) * logit_scale

        # Constraint Step 4: Apply Softmax
        probs = softmax(logits, axis=1)
        preds = np.argmax(probs, axis=1) # Shape: (1748,)

        # Calculate confusion matrix and add to the family's aggregate pool
        cm = confusion_matrix(y_true, preds, labels=range(5))
        agg_cm += cm

    # Calculate percentages for the annotations
    total_preds = agg_cm.sum()
    cm_perc = (agg_cm / total_preds) * 100

    # Create custom string annotations: "Raw Count \n (Percentage%)"
    annot_labels = np.empty_like(agg_cm, dtype=object)
    for i in range(5):
        for j in range(5):
            annot_labels[i, j] = f"{agg_cm[i, j]}\n({cm_perc[i, j]:.1f}%)"

    # Render the heatmap
    sns.heatmap(agg_cm, annot=annot_labels, fmt='', cmap='YlGnBu', cbar=False,
                xticklabels=grade_order, yticklabels=grade_order, ax=axes[idx],
                annot_kws={"size": 13})

    axes[idx].set_title(f'Prompt Family: {fam}', pad=15, fontweight='bold')
    axes[idx].set_xlabel('Predicted DR Grade')
    axes[idx].set_ylabel('True DR Grade')

plt.suptitle('Aggregated Confusion Matrices Across Prompt Variations', y=1.02, fontsize=22, fontweight='bold')
plt.tight_layout()

# Save as high-res PNG and PDF for LaTeX
plt.savefig('new_thesis_confusion_matrices.png', dpi=600, bbox_inches='tight')
plt.savefig('new_thesis_confusion_matrices.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ==========================================
# THESIS BLOCK 2: Confidence Density Distributions (KDE)
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax

# Ensure aesthetic context is set for thesis inclusion
sns.set_context("talk")
sns.set_style("whitegrid")

# Define necessary iterables (should carry over from previous block, but defined for safety)
families = ['T_label', 'T_expert', 'T_layman', 'T_exclusion']
grade_order = ['noDR', 'mildDR', 'modDR', 'sevDR', 'prolDR']

# Prepare the 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, fam in enumerate(families):
    correct_confs = []
    incorrect_confs = []

    # Iterate through all 10 variations to build a comprehensive distribution pool
    for var_num in range(1, 11):
        # Extract the 5 text embeddings for this specific variation
        t_sub = np.zeros((5, 512))
        for g_idx, grade in enumerate(grade_order):
            mask = (text_meta['Family'] == fam) & (text_meta['Variation_Number'] == var_num) & (text_meta['Grade'] == grade)
            t_idx = text_meta.index[mask].tolist()[0]
            t_sub[g_idx] = text_emb[t_idx]

        # Constraint Step 1b: L2 Normalize Text Embeddings
        t_norm = t_sub / np.linalg.norm(t_sub, axis=1, keepdims=True)

        # Constraint Step 2 & 3: Cosine Similarity multiplied by Logit Scale
        logits = (v_norm @ t_norm.T) * logit_scale

        # Constraint Step 4: Apply Softmax to get probabilities
        probs = softmax(logits, axis=1)

        # Extract predictions and maximum confidence scores
        preds = np.argmax(probs, axis=1)
        confs = np.max(probs, axis=1)

        # Separate into correct and incorrect groups
        correct_mask = (preds == y_true)
        correct_confs.extend(confs[correct_mask])
        incorrect_confs.extend(confs[~correct_mask])

    # Plot KDE for Correct Predictions
    sns.kdeplot(correct_confs, ax=axes[idx], fill=True, color='green', alpha=0.5, label='Correct Predictions')

    # Plot KDE for Incorrect Predictions
    sns.kdeplot(incorrect_confs, ax=axes[idx], fill=True, color='crimson', alpha=0.5, label='Incorrect Predictions')

    # Formatting and strict axis constraints
    axes[idx].set_title(f'Prompt Family: {fam}', pad=15, fontweight='bold')
    axes[idx].set_xlabel('Model Confidence (0.0 to 1.0)')
    axes[idx].set_ylabel('Density')
    axes[idx].set_xlim(0.0, 1.0)

    # Only add legend to the first plot to keep the grid clean
    if idx == 0:
        axes[idx].legend(loc='upper left', frameon=True)

plt.suptitle('Model Calibration: Confidence Distributions of Correct vs. Incorrect Predictions', y=1.03, fontsize=22, fontweight='bold')
plt.tight_layout()

# Save as high-res PNG and PDF for LaTeX
plt.savefig('new_thesis_confidence_density.png', dpi=300, bbox_inches='tight')
plt.savefig('new_thesis_confidence_density.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ==========================================
# THESIS BLOCK 3: Variance & Calibration Table (ECE)
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
from scipy.special import softmax

# --- 1. Aesthetics & Setup ---
sns.set_context("talk")

# --- 2. Load Data ---
vision_emb = np.load('cached_vision_embeddings.npy')
text_emb = np.load('cached_text_embeddings_custom.npy')
logit_scale = float(np.load('flair_logit_scale.npy'))
vision_meta = pd.read_csv('cached_vision_metadata.csv')
text_meta = pd.read_csv('cached_text_metadata_custom.csv')

# Preprocess vision metadata
valid_idx = vision_meta['adjudicated_dr_grade'].notna()
vision_emb = vision_emb[valid_idx]
vision_meta = vision_meta[valid_idx].reset_index(drop=True)
y_true = vision_meta['adjudicated_dr_grade'].astype(int).values

# Constraint Step 1a: L2 Normalize Vision Embeddings
v_norm = vision_emb / np.linalg.norm(vision_emb, axis=1, keepdims=True)

families = ['T_label', 'T_expert', 'T_layman', 'T_exclusion']
grade_order = ['noDR', 'mildDR', 'modDR', 'sevDR', 'prolDR']

# --- 3. Mathematical Functions ---
def calculate_ece(probs, labels, preds, n_bins=10):
    """Calculates the Expected Calibration Error (ECE) using 10 equally spaced bins."""
    confidences = np.max(probs, axis=1)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    ece = 0.0
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.mean()
        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(preds[in_bin] == labels[in_bin])
            avg_confidence_in_bin = np.mean(confidences[in_bin])
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    return ece

# --- 4. Computation Loop ---
variation_results = []

for fam in families:
    for var_num in range(1, 11):
        t_sub = np.zeros((5, 512))
        for g_idx, grade in enumerate(grade_order):
            mask = (text_meta['Family'] == fam) & (text_meta['Variation_Number'] == var_num) & (text_meta['Grade'] == grade)
            t_idx = text_meta.index[mask].tolist()[0]
            t_sub[g_idx] = text_emb[t_idx]

        # Strict Constraint Pipeline
        t_norm = t_sub / np.linalg.norm(t_sub, axis=1, keepdims=True)
        logits = (v_norm @ t_norm.T) * logit_scale
        probs = softmax(logits, axis=1)
        preds = np.argmax(probs, axis=1)

        # Calculate Metrics
        acc = accuracy_score(y_true, preds)
        ece = calculate_ece(probs, y_true, preds, n_bins=10)

        variation_results.append({
            'Family': fam,
            'Variation_Number': var_num,
            'Accuracy': acc,
            'ECE': ece
        })

# --- 5. Data Aggregation & Output ---
df_all = pd.DataFrame(variation_results)
df_all.to_csv('thesis_prompt_variance.csv', index=False)
print("Saved full variance data to 'thesis_prompt_variance.csv'.")

# Calculate Family-level Mean and Std Dev
df_summary = df_all.groupby('Family').agg({
    'Accuracy': ['mean', 'std'],
    'ECE': ['mean', 'std']
}).reset_index()

# Flatten hierarchical columns for the table
df_summary.columns = ['Prompt Family', 'Accuracy (Mean)', 'Accuracy (SD)', 'ECE (Mean)', 'ECE (SD)']

# Format standard deviations and means for a clean presentation
for col in ['Accuracy (Mean)', 'Accuracy (SD)', 'ECE (Mean)', 'ECE (SD)']:
    df_summary[col] = df_summary[col].apply(lambda x: f"{x:.4f}")

# Combine Mean and SD into a single string for the thesis table
df_thesis_table = pd.DataFrame()
df_thesis_table['Prompt Family'] = df_summary['Prompt Family']
df_thesis_table['Accuracy (Mean ± SD)'] = df_summary['Accuracy (Mean)'] + " ± " + df_summary['Accuracy (SD)']
df_thesis_table['ECE (Mean ± SD)'] = df_summary['ECE (Mean)'] + " ± " + df_summary['ECE (SD)']

# --- 6. Plotting the Table Visual ---
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

# Render table
table = ax.table(cellText=df_thesis_table.values,
                 colLabels=df_thesis_table.columns,
                 cellLoc='center',
                 loc='center')

# Style the table
table.auto_set_font_size(False)
table.set_fontsize(14)
table.scale(1.2, 2.0)

# Make column headers bold and apply a subtle color
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#4C72B0') # Consistent with the seaborn default palette
    else:
        cell.set_facecolor('#F8F9FA')

plt.title('Summary of Prompt Sensitivity and Calibration across Families', pad=20, fontweight='bold')
plt.tight_layout()

# Save for LaTeX thesis
plt.savefig('thesis_variance_summary.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
!pip install umap-learn

# ==========================================
# THESIS BLOCK 4: UMAP 2D Multimodal Projection
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap

# --- 1. Aesthetics & Setup ---
sns.set_context("talk")
sns.set_style("whitegrid")

# --- 2. Load Data (Ensuring Memory State is Clean) ---
vision_emb = np.load('cached_vision_embeddings.npy')
text_emb = np.load('cached_text_embeddings_custom.npy')
vision_meta = pd.read_csv('cached_vision_metadata.csv')
text_meta = pd.read_csv('cached_text_metadata_custom.csv')

# Clean vision metadata
valid_idx = vision_meta['adjudicated_dr_grade'].notna()
vision_emb = vision_emb[valid_idx]
vision_meta = vision_meta[valid_idx].reset_index(drop=True)
y_true = vision_meta['adjudicated_dr_grade'].astype(int).values

grade_order = ['noDR', 'mildDR', 'modDR', 'sevDR', 'prolDR']
families = ['T_label', 'T_expert', 'T_layman', 'T_exclusion']

# --- 3. Mathematical Constraint: L2 Normalization ---
# UMAP will use 'cosine' metric, so we normalize our inputs exactly as the model does
v_norm = vision_emb / np.linalg.norm(vision_emb, axis=1, keepdims=True)
t_norm = text_emb / np.linalg.norm(text_emb, axis=1, keepdims=True)

# Combine vision and text into a single manifold space for projection
combined_emb = np.vstack([v_norm, t_norm])

# --- 4. Fit UMAP ---
# We use metric='cosine' to mirror the dot-product similarity mechanism of FLAIR
reducer = umap.UMAP(n_components=2, metric='cosine', random_state=42, n_neighbors=15, min_dist=0.1)
combined_2d = reducer.fit_transform(combined_emb)

# Split back out into Vision and Text 2D coordinates
v_2d = combined_2d[:len(v_norm)]
t_2d = combined_2d[len(v_norm):]

# --- 5. Plotting ---
fig, ax = plt.subplots(figsize=(14, 10))

# Distinct 5-color palette for the DR Grades
color_palette = sns.color_palette("colorblind", 5)

# 5A. Plot Vision Embeddings (Small, semi-transparent dots)
for i, grade in enumerate(grade_order):
    idx = np.where(y_true == i)[0]
    ax.scatter(v_2d[idx, 0], v_2d[idx, 1],
               color=color_palette[i],
               alpha=0.4, s=30, edgecolors='none',
               label=f'Vision: {grade}')

# 5B. Plot Text Embeddings (Large markers, black edges, distinct shapes per family)
# We use different markers because color is mapped to the Grade
family_markers = {'T_label': '*',       # Star
                  'T_expert': 'D',      # Diamond
                  'T_layman': 'X',      # Cross
                  'T_exclusion': 'p'}   # Pentagon

for fam in families:
    for i, grade in enumerate(grade_order):
        # Find all 10 variations for this specific family and grade
        mask = (text_meta['Family'] == fam) & (text_meta['Grade'] == grade)
        idx = np.where(mask)[0]

        # Only add the label to the legend ONCE per family (using the first grade loop)
        label = f'Text: {fam}' if i == 0 else None

        ax.scatter(t_2d[idx, 0], t_2d[idx, 1],
                   color=color_palette[i],
                   marker=family_markers[fam],
                   s=350, edgecolors='black', linewidth=1.5, alpha=0.95,
                   label=label)

ax.set_title('UMAP Projection: Vision-Language Alignment Space', pad=20, fontweight='bold')
ax.set_xlabel('UMAP Dimension 1')
ax.set_ylabel('UMAP Dimension 2')

# --- 6. Legend Organization ---
# Extract handles and labels to clean up the legend layout
handles, labels = ax.get_legend_handles_labels()
vision_handles = handles[:5]
vision_labels = labels[:5]
text_handles = handles[5:]
text_labels = labels[5:]

# Create a clean, out-of-plot legend separating Vision and Text
legend1 = ax.legend(vision_handles, vision_labels, title='Images (True Grade)',
                    bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
legend2 = ax.legend(text_handles, text_labels, title='Prompts (Family Shape)',
                    bbox_to_anchor=(1.02, 0.65), loc='upper left', frameon=False, scatterpoints=1)
ax.add_artist(legend1) # Add vision legend back

plt.tight_layout()

# Save as high-res PNG and PDF for LaTeX
plt.savefig('new_thesis_umap_projection.png', dpi=300, bbox_inches='tight')
plt.savefig('new_thesis_umap_projection.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ==========================================
# THESIS BLOCK 5: Per-Class Metrics & F1 Heatmap
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support
from scipy.special import softmax

# --- 1. Aesthetics & Setup ---
sns.set_context("talk")
sns.set_style("white")

# --- 2. Load Data ---
vision_emb = np.load('cached_vision_embeddings.npy')
text_emb = np.load('cached_text_embeddings_custom.npy')
logit_scale = float(np.load('flair_logit_scale.npy'))
vision_meta = pd.read_csv('cached_vision_metadata.csv')
text_meta = pd.read_csv('cached_text_metadata_custom.csv')

# Preprocess vision metadata
valid_idx = vision_meta['adjudicated_dr_grade'].notna()
vision_emb = vision_emb[valid_idx]
vision_meta = vision_meta[valid_idx].reset_index(drop=True)
y_true = vision_meta['adjudicated_dr_grade'].astype(int).values

# Constraint Step 1a: L2 Normalize Vision Embeddings
v_norm = vision_emb / np.linalg.norm(vision_emb, axis=1, keepdims=True)

families = ['T_label', 'T_expert', 'T_layman', 'T_exclusion']
grade_order = ['noDR', 'mildDR', 'modDR', 'sevDR', 'prolDR']

# --- 3. Compute Aggregated Predictions ---
results = []

for fam in families:
    preds_agg = []
    y_true_agg = []

    for var_num in range(1, 11):
        t_sub = np.zeros((5, 512))
        for g_idx, grade in enumerate(grade_order):
            mask = (text_meta['Family'] == fam) & (text_meta['Variation_Number'] == var_num) & (text_meta['Grade'] == grade)
            t_idx = text_meta.index[mask].tolist()[0]
            t_sub[g_idx] = text_emb[t_idx]

        # Strict Constraint Pipeline
        t_norm = t_sub / np.linalg.norm(t_sub, axis=1, keepdims=True)
        logits = (v_norm @ t_norm.T) * logit_scale
        probs = softmax(logits, axis=1)
        preds = np.argmax(probs, axis=1)

        # Pool all predictions across variations to get an aggregate performance
        preds_agg.extend(preds)
        y_true_agg.extend(y_true)

    # Calculate Precision, Recall, and F1 for the 5 classes using sklearn
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_agg, preds_agg, labels=[0, 1, 2, 3, 4], zero_division=0
    )

    for i, grade in enumerate(grade_order):
        results.append({
            'Family': fam,
            'Grade': grade,
            'Precision': precision[i],
            'Recall': recall[i],
            'F1-Score': f1[i]
        })

# --- 4. Format Data Output ---
df_metrics = pd.DataFrame(results)

# Create a nested DataFrame (MultiIndex) for the CSV export
df_nested = df_metrics.set_index(['Family', 'Grade'])
df_nested.to_csv('thesis_per_class_metrics.csv')
print("Successfully saved nested metrics to 'thesis_per_class_metrics.csv'.")

# --- 5. Generate Heatmap ---
# Pivot specifically for the F1-Score heatmap matrix
df_f1 = df_metrics.pivot(index='Family', columns='Grade', values='F1-Score')
df_f1 = df_f1[grade_order] # Guarantee specific column order from noDR to prolDR

plt.figure(figsize=(10, 6))
# Using 'magma' as it visually emphasizes severe drops in performance (dark colors = bad performance)
sns.heatmap(df_f1, annot=True, fmt='.3f', cmap='magma', vmin=0.0, vmax=1.0,
            linewidths=1.5, linecolor='white')

plt.title('F1-Score Heatmap by Prompt Family and DR Grade', pad=20, fontweight='bold')
plt.xlabel('DR Grade')
plt.ylabel('Prompt Family')
plt.yticks(rotation=0)
plt.tight_layout()

# Save for LaTeX thesis
plt.savefig('new_thesis_f1_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('new_thesis_f1_heatmap.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ==========================================
# THESIS BLOCK 6: Publication-Quality Reliability Diagrams
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax
from sklearn.metrics import accuracy_score

# --- 1. Aesthetics & Setup ---
sns.set_context("talk")
sns.set_style("whitegrid")

# --- 2. Load Data (Ensuring Memory State is Clean) ---
vision_emb = np.load('cached_vision_embeddings.npy')
text_emb = np.load('cached_text_embeddings_custom.npy')
logit_scale = float(np.load('flair_logit_scale.npy'))
vision_meta = pd.read_csv('cached_vision_metadata.csv')
text_meta = pd.read_csv('cached_text_metadata_custom.csv')

# Clean vision metadata
valid_idx = vision_meta['adjudicated_dr_grade'].notna()
vision_emb = vision_emb[valid_idx]
vision_meta = vision_meta[valid_idx].reset_index(drop=True)
y_true = vision_meta['adjudicated_dr_grade'].astype(int).values

# Constraint Step 1a: L2 Normalize Vision Embeddings
v_norm = vision_emb / np.linalg.norm(vision_emb, axis=1, keepdims=True)

families = ['T_label', 'T_expert', 'T_layman', 'T_exclusion']
grade_order = ['noDR', 'mildDR', 'modDR', 'sevDR', 'prolDR']
n_bins = 10
bin_boundaries = np.linspace(0, 1.0, n_bins + 1)

# --- 3. Figure Setup ---
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, fam in enumerate(families):
    preds_agg = []
    confs_agg = []
    y_true_agg = []

    # Pool all 10 variations (17,480 total predictions)
    for var_num in range(1, 11):
        t_sub = np.zeros((5, 512))
        for g_idx, grade in enumerate(grade_order):
            mask = (text_meta['Family'] == fam) & (text_meta['Variation_Number'] == var_num) & (text_meta['Grade'] == grade)
            t_idx = text_meta.index[mask].tolist()[0]
            t_sub[g_idx] = text_emb[t_idx]

        # Strict Constraint Pipeline
        t_norm = t_sub / np.linalg.norm(t_sub, axis=1, keepdims=True)
        logits = (v_norm @ t_norm.T) * logit_scale
        probs = softmax(logits, axis=1)

        preds_agg.extend(np.argmax(probs, axis=1))
        confs_agg.extend(np.max(probs, axis=1))
        y_true_agg.extend(y_true)

    preds_agg = np.array(preds_agg)
    confs_agg = np.array(confs_agg)
    y_true_agg = np.array(y_true_agg)

    # --- 4. Calculate Calibration Metrics ---
    bin_accs = []
    bin_confs = []
    bin_props = []
    ece = 0.0

    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i+1]

        # Identify samples in this bin
        in_bin = (confs_agg > bin_lower) & (confs_agg <= bin_upper)
        prop_in_bin = in_bin.mean()
        bin_props.append(prop_in_bin)

        if prop_in_bin > 0:
            acc_in_bin = np.mean(preds_agg[in_bin] == y_true_agg[in_bin])
            conf_in_bin = np.mean(confs_agg[in_bin])

            bin_accs.append(acc_in_bin)
            bin_confs.append(conf_in_bin)
            ece += np.abs(conf_in_bin - acc_in_bin) * prop_in_bin
        else:
            bin_accs.append(np.nan)
            bin_confs.append(np.nan)

    overall_acc = accuracy_score(y_true_agg, preds_agg)
    overall_conf = np.mean(confs_agg)

    # --- 5. Plotting on Primary Axis ---
    ax = axes[idx]

    # Perfect calibration baseline
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration', zorder=1)

    # Calibration Curve (Color logic based on prompt)
    curve_color = 'teal' if fam == 'T_label' else 'crimson'
    ax.plot(bin_confs, bin_accs, marker='o', markersize=8, color=curve_color,
            linewidth=2.5, label='Model Calibration', zorder=3)

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Family: {fam}', pad=10, fontweight='bold')

    # --- 6. Plotting Secondary Axis (Histogram) ---
    ax2 = ax.twinx()
    # Bin centers for the histogram bars
    bin_centers = bin_boundaries[:-1] + (1.0 / n_bins) / 2
    # Adjust bar height scale so it stays in the bottom ~30% of the graph
    max_prop = max(bin_props) if max(bin_props) > 0 else 1.0
    ax2.bar(bin_centers, bin_props, width=(1.0 / n_bins)*0.8, color='slategray',
            alpha=0.3, zorder=0, label='% of Samples')
    ax2.set_ylim(0.0, max_prop * 3.5) # Force bars to sit at the bottom
    ax2.set_yticks([]) # Hide secondary y-axis ticks to keep it clean
    ax2.set_ylabel('')

    # --- 7. Metrics Text Box ---
    stats_text = (
        f"Overall Acc:  {overall_acc*100:.1f}%\n"
        f"Overall ECE:  {ece*100:.1f}%\n"
        f"Avg Conf:     {overall_conf*100:.1f}%"
    )
    # Placed in the upper left, slightly offset
    ax.text(0.04, 0.96, stats_text, transform=ax.transAxes, fontsize=13,
            verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='lightgray'),
            zorder=4)

plt.suptitle('Impact of Lexical Variation on Model Calibration (Reliability Diagrams)', y=1.03, fontsize=22, fontweight='bold')
plt.tight_layout()

# Save as high-res PNG and PDF for LaTeX
plt.savefig('new_thesis_reliability_diagrams_v2.png', dpi=300, bbox_inches='tight')
plt.savefig('new_thesis_reliability_diagrams_v2.pdf', format='pdf', bbox_inches='tight')
plt.show()

FIGURES FOR LIT REV


In [ ]:
# ==============================================================================
# Python Script to Generate Figure 2.1: Methodological Flaws Chart
# Save this script as `generate_flaws_chart.py` and run it to produce
# the `methodological_flaws_chart.pdf` referenced in your LaTeX document.
# Required packages: pip install matplotlib seaborn numpy
# ==============================================================================

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Set the academic aesthetic style for the plot to match the thesis results
sns.set_context("talk")
sns.set_style("white")
plt.rcParams.update({
    'pdf.fonttype': 42  # Ensures fonts are embedded in PDF
})

# Total number of audited papers
total_papers = 127

# Data derived directly from the forensic audit of the 127 cited papers
categories = [
    'Biological Data Leakage\n(Image-Level Splitting)',
    'Test Set Mutilation\n(Filtering / Sanitizing)',
    'Pre-Split Statistical Leakage\n(Global Preprocessing)',
    'Task Simplification\n(Scope Bait-and-Switch)',
    'Distributional Contamination\n(Unified Mixed Pooling)'
]
counts = [47, 18, 15, 15, 12]

# Dynamically calculate percentages based on the total number of papers
percentages = [round((count / total_papers) * 100, 1) for count in counts]

# Create the figure
fig, ax = plt.subplots(figsize=(10, 6.5))

# Extract exactly 5 distinct colors from the YlGnBu palette
colors = sns.color_palette("YlGnBu", len(categories))

# Create horizontal bar plot
bars = ax.barh(categories, percentages, color=colors, edgecolor='black', height=0.6, linewidth=1.2)

# Invert y-axis to have the highest percentage at the top
ax.invert_yaxis()

# Customize axes and labels
ax.set_xlabel(f'Prevalence in Audited Literature (%) [N={total_papers}]',
              fontsize=13, weight='bold', labelpad=15)
ax.set_title('Frequency of Critical Methodological Flaws in\nRecent DR Deep Learning Literature',
             fontsize=15, weight='bold', pad=20)

# Adjusted x-limit from 105 down to 45 to better fit the maximum percentage (~37.0%)
ax.set_xlim(0, 45)

# Add percentage and raw n-count labels to the end of each bar
for bar, count, pct in zip(bars, counts, percentages):
    width = bar.get_width()
    ax.text(width + 0.5, bar.get_y() + bar.get_height()/2,
            f'{pct}% (n={count})',
            va='center', ha='left', fontsize=12, weight='bold', color='#2C3E50')

# Clean up the grid and spines for a crisp LaTeX aesthetic
ax.xaxis.grid(True, linestyle='--', alpha=0.7, color='#BDBDBD')
ax.yaxis.grid(False)

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

# Remove y-axis tick marks for cleaner look
ax.tick_params(axis='y', length=0)

# Tweak layout for perfect rendering
plt.tight_layout()

# Save the figure as a high-resolution PDF for seamless LaTeX inclusion
output_filename = 'methodological_flaws_chart.pdf'
plt.savefig(output_filename, format='pdf', dpi=600, bbox_inches='tight')
print(f"Chart generated successfully: {output_filename}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Match the thesis results visualization style
sns.set_context("talk")
sns.set_style("white")
plt.rcParams.update({
    'pdf.fonttype': 42
})
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 1.2

# Data extracted directly from the forensic audit of the literature
studies = [
    "Sharma et al.\n(DiaRetDB1)",
    "Chavan et al.\n(IDRID)",
    "Yang et al.\n(ORIGA)",
    "Alsheikhy et al.\n(Paraguay)",
    "Poranki et al.\n(EyePACS PDR/Sev)"
]

# Total original images available in the referenced datasets/classes
actual_dataset_size = [89, 516, 650, 757, 1581] # Poranki combined 708 + 873

# Inflated test-set sizes reported by the authors (proving leakage/augmentation)
reported_test_size = [303, 535, 2200, 988, 2842] # Poranki combined 1437 + 1405

# X-axis locations
x = np.arange(len(studies))
width = 0.35  # Width of the bars

# Create figure
fig, ax = plt.subplots(figsize=(10, 6.5))

# Extract distinct, cohesive colors from the YlGnBu palette
pal = sns.color_palette("YlGnBu", 6)

rects1 = ax.bar(x - width/2, actual_dataset_size, width,
                label='Actual Total Dataset Size',
                color=pal[5], edgecolor='black', linewidth=1)

rects2 = ax.bar(x + width/2, reported_test_size, width,
                label='Reported Test Set Size\n(Proves Leakage/Tampering)',
                color=pal[2], edgecolor='black', linewidth=1, hatch='//')

# Add labels, title, and formatting
ax.set_ylabel('Number of Images', fontweight='bold', labelpad=12)
ax.set_title('Test Set Contamination: The >100% Problem in Recent Literature',
             fontweight='bold', fontsize=14, pad=20)
ax.set_xticks(x)
ax.set_xticklabels(studies, fontweight='bold', fontsize=11)
ax.legend(loc='upper left', frameon=True, shadow=True, fancybox=True)

# Function to auto-label bars with their exact numerical values
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{int(height)}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 4),  # 4 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

autolabel(rects1)
autolabel(rects2)

# Draw a subtle line/arrow indicating where the math breaks (Test > Total)
for i in range(len(studies)):
    ax.annotate('', xy=(x[i] + width/2, reported_test_size[i] + 50),
                xytext=(x[i] - width/2, actual_dataset_size[i] + 50),
                arrowprops=dict(arrowstyle="<-", color=pal[4], lw=2, ls='-'))

# Clean up layout
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()

# Save as PDF for high-quality LaTeX inclusion
plt.savefig('test_set_contamination.pdf', dpi=300, bbox_inches='tight')
print("Chart successfully saved as 'test_set_contamination.pdf'")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set the style for an academic paper
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.family': 'serif', 'font.size': 12})

# Data extracted from FLAIR paper (Silva-Rodríguez et al., 2025) Table 4
# Metric: Average Class Accuracy (ACA) across unseen datasets
models = ['CLIP\n(Generalist)', 'BiomedCLIP\n(Medical Generalist)', 'FLAIR\n(Retina Specialist)']

# Naive Prompts (e.g., "cataract")
naive_prompts = [0.346, 0.501, 0.400]

# Expert Knowledge Prompts (e.g., "opacity in the macular area")
expert_prompts = [0.332, 0.491, 0.683]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5.5))

# Plot bars
rects1 = ax.bar(x - width/2, naive_prompts, width, label='Naïve Prompts (Class Name)',
                color='#a6bddb', edgecolor='black', linewidth=1.2)
rects2 = ax.bar(x + width/2, expert_prompts, width, label='Expert Knowledge Prompts (AAO)',
                color='#1c9099', edgecolor='black', linewidth=1.2)

# Axis formatting
ax.set_ylabel('Average Class Accuracy (ACA)', fontweight='bold')
ax.set_title('Impact of Expert Prompts on Zero-Shot Generalization', fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models, fontweight='bold')
ax.set_ylim(0, 0.8)

# Add subtle shading to visually separate and emphasize FLAIR
ax.axvspan(1.5, 2.5, color='lightgray', alpha=0.3, zorder=0)

# Legend
ax.legend(loc='upper left', frameon=True, shadow=True)

# Add value labels
def autolabel(rects, is_expert=False):
    for i, rect in enumerate(rects):
        height = rect.get_height()

        # Add delta text for expert prompts to highlight the drop or gain
        if is_expert:
            diff = expert_prompts[i] - naive_prompts[i]
            color = 'green' if diff > 0 else 'red'
            sign = '+' if diff > 0 else ''
            ax.annotate(f'({sign}{diff:.3f})',
                        xy=(rect.get_x() + rect.get_width() / 2, height + 0.04),
                        ha='center', va='bottom', fontweight='bold', color=color, fontsize=10)

        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontweight='bold', fontsize=11)

autolabel(rects1)
autolabel(rects2, is_expert=True)

# Finalize layout and save
plt.tight_layout()
plt.savefig('flair_zeroshot_comparison.pdf', format='pdf', dpi=300, bbox_inches='tight')
print("Figure successfully saved as 'flair_zeroshot_comparison.pdf'")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ==============================================================================
# Python Script to Generate Figure 1: The Synthetic Paradox and Metric Inflation
# Run this script to generate 'synthetic_paradox_metric_inflation.pdf'
# Requires: matplotlib, numpy
# ==============================================================================

# Apply strict academic styling
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 11

# Data extracted from forensic audit literature review
categories = [
    "Hu et al. (2026)\nMicroaneurysm Acc.\n[Leaked]",
    "Haq et al. (2024)\nGlobal Acc.\n[Mixed Pools]",
    "MRizvana et al. (2024)\nGlobal Acc.\n[No Patient Split]",
    "Sun et al. (2025)\nTrue PDR Recall\n[Rigorous Baseline]"
]

# Accuracy percentages based on text file reports
values = [99.99, 99.66, 99.28, 18.18]

# Define colors: Crimson for flawed/inflated metrics, Dark Blue for the realistic baseline
colors = ['#c0392b', '#c0392b', '#c0392b', '#2980b9']

fig, ax = plt.subplots(figsize=(9, 6))

# Create the bar plot
bars = ax.bar(categories, values, color=colors, edgecolor='black', linewidth=1.2, width=0.6)

# Add value labels on top of the bars
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}%',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5),  # 5 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom', fontweight='bold', fontsize=11)

# Formatting the axes
ax.set_ylabel('Reported Performance Metric (%)', fontweight='bold')
ax.set_title('The Synthetic Paradox: Metric Inflation vs. True Minority Class Generalization',
             fontweight='bold', pad=20)
ax.set_ylim(0, 115)  # Leave room for labels

# Adding a horizontal threshold line to show typical clinical approval requirements
ax.axhline(85, color='dimgray', linestyle='--', linewidth=1.5, alpha=0.8, label='Typical Minimum Screening Threshold (85%)')
ax.legend(loc='upper right', frameon=True, edgecolor='black')

# Customizing spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add subtle horizontal gridlines
ax.yaxis.grid(True, linestyle=':', alpha=0.7)
ax.xaxis.grid(False)

# Add a subtle background shading to highlight the disparity
ax.axvspan(-0.5, 2.5, color='#e74c3c', alpha=0.05, lw=0)
ax.axvspan(2.5, 3.5, color='#3498db', alpha=0.05, lw=0)

plt.tight_layout()

# Save as PDF for high-quality LaTeX rendering
plt.savefig('synthetic_paradox_metric_inflation.pdf', format='pdf', dpi=300, bbox_inches='tight')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load data
embeddings = np.load('cached_text_embeddings_custom.npy')
metadata = pd.read_csv('cached_text_metadata_custom.csv')

# 2. Add original index mapping
metadata['orig_idx'] = metadata.index

# 3. Define the deterministic family order for the axes
family_order = ['T_label', 'T_expert', 'T_layman', 'T_exclusion']
metadata['Family'] = pd.Categorical(metadata['Family'], categories=family_order, ordered=True)

# 4. Sort metadata hierarchically (Family -> Grade -> Variation)
metadata_sorted = metadata.sort_values(by=['Family', 'Grade', 'Variation_Number']).dropna(subset=['Family'])
sorted_indices = metadata_sorted['orig_idx'].values

# 5. Reorder embeddings to group by Family
embeddings_sorted = embeddings[sorted_indices]

# 6. Compute 200x200 pairwise cosine similarity matrix
sim_matrix = cosine_similarity(embeddings_sorted)

# 7. Generate High-Resolution Heatmap
plt.figure(figsize=(12, 10))
ax = sns.heatmap(sim_matrix, cmap='viridis', vmin=-1, vmax=1,
                 xticklabels=False, yticklabels=False)

# 8. Calculate exact cluster boundaries for demarcations
family_counts = metadata_sorted['Family'].value_counts(sort=False).dropna()
boundaries = [0] + list(family_counts.cumsum())

# Add solid white demarcations
for b in boundaries[1:-1]:
    ax.axhline(b, color='white', lw=1.5)
    ax.axvline(b, color='white', lw=1.5)

# Add centered labels for each group
ticks = []
for i in range(len(boundaries)-1):
    ticks.append((boundaries[i] + boundaries[i+1]) / 2)

ax.set_xticks(ticks)
ax.set_xticklabels(family_counts.index, rotation=45, ha='right', fontsize=14)
ax.set_yticks(ticks)
ax.set_yticklabels(family_counts.index, rotation=0, va='center', fontsize=14)

plt.title("Pairwise Cosine Similarity Matrix ($200 \\times 200$) by Prompt Family", fontsize=16)
plt.tight_layout()

# 9. Save artifacts
plt.savefig('new_similarity_heatmap.png', dpi=600)
plt.savefig('new_similarity_heatmap.pdf', dpi=600)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax

def main():
    # Load embeddings and logit scale scalar
    vision_embeddings = np.load('cached_vision_embeddings.npy')  # Shape: (1748, 512)
    text_embeddings = np.load('cached_text_embeddings_custom.npy') # Shape: (200, 512)
    logit_scale = np.load('flair_logit_scale.npy')

    # Extract scalar if wrapped in a zero-dimensional array
    if logit_scale.ndim > 0:
        logit_scale = logit_scale.item()

    # Apply L2 normalization to ensure mathematically bounded cosine similarity [-1, 1]
    vision_norm = vision_embeddings / np.linalg.norm(vision_embeddings, axis=1, keepdims=True)
    text_norm = text_embeddings / np.linalg.norm(text_embeddings, axis=1, keepdims=True)

    # Compute raw cosine similarity matrix (1748, 200)
    # Each row i represents the similarities between image i and all 200 text queries
    raw_similarity = vision_norm @ text_norm.T

    # Extract the top-1 prediction confidence before scaling (max similarity per image)
    max_raw_sim = raw_similarity.max(axis=1)

    # Apply the learned temperature parameter (s = 17.4355)
    # followed by the softmax function along the text dimension (j)
    scaled_similarity = raw_similarity * logit_scale
    probabilities = softmax(scaled_similarity, axis=1)

    # Extract the calibrated maximum probability per image
    max_probs = probabilities.max(axis=1)

    # Initialize plotting environment
    sns.set_theme(style="whitegrid", context="paper")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left Panel: Raw Cosine Similarity KDE
    sns.kdeplot(max_raw_sim, ax=axes[0], fill=True, color='#2c7bb6', linewidth=2, cut=0)
    axes[0].set_title('Raw Cosine Similarity Distribution\n(Uncalibrated Top-1 Confidence)', fontsize=12, pad=10)
    axes[0].set_xlabel('Max Cosine Similarity $S(v_i, t_j)$', fontsize=11)
    axes[0].set_ylabel('Density', fontsize=11)

    # Right Panel: Calibrated Softmax Probability KDE
    sns.kdeplot(max_probs, ax=axes[1], fill=True, color='#d7191c', linewidth=2, cut=0)
    axes[1].set_title('Calibrated Probability Distribution\n(Post-Scaling Softmax)', fontsize=12, pad=10)
    axes[1].set_xlabel('Max Softmax Probability $P(y|v_i)$', fontsize=11)
    axes[1].set_ylabel('Density', fontsize=11)

    plt.suptitle('Density Shift via Temperature Scaling in Vision-Language Latent Space', fontsize=14, y=1.05)
    plt.tight_layout()

    # Save artifact formats standard for academic dissemination
    plt.savefig('new_scaling_effect.png', dpi=300, bbox_inches='tight')
    plt.savefig('new_scaling_effect.pdf', format='pdf', bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import matplotlib.pyplot as plt

grades = ['noDR\n(0)', 'mildDR\n(1)', 'modDR\n(2)', 'sevDR\n(3)', 'prolDR\n(4)', 'ungradeable']
counts = [1017, 270, 347, 75, 35, 4]

plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 6))
bars = plt.bar(grades, counts, color='#4c72b0', edgecolor='black', width=0.6)

for bar in bars:
    height = bar.get_height()
    plt.annotate(f'n={height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontweight='bold')

plt.ylabel('Number of Images', fontsize=12, fontweight='bold')
plt.xlabel('Diabetic Retinopathy Severity Grade', fontsize=12, fontweight='bold')
plt.title('MESSIDOR-2 Dataset Categorical Distribution (N=1,748)', fontsize=14, fontweight='bold')
plt.ylim(0, 1200)
plt.tight_layout()
plt.savefig('messidor2_dist.pdf', format='pdf', dpi=600)
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# 1. CORE MATH: EXPECTED CALIBRATION ERROR
# ==========================================
def calculate_ece(confidences, accuracies, num_bins=10):
    """
    Calculates the Expected Calibration Error (ECE).
    Divides confidence scores into M bins and computes the weighted average
    of the absolute difference between accuracy and confidence in each bin.
    """
    bin_boundaries = np.linspace(0, 1, num_bins + 1)
    ece = 0.0

    for i in range(num_bins):
        # Identify predictions that fall into this specific confidence bin
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i+1]

        # Include the upper bound exactly for the final bin
        if i == num_bins - 1:
            in_bin = (confidences >= bin_lower) & (confidences <= bin_upper)
        else:
            in_bin = (confidences >= bin_lower) & (confidences < bin_upper)

        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(accuracies[in_bin])
            avg_confidence_in_bin = np.mean(confidences[in_bin])
            # The absolute difference between what it thinks and what it knows
            ece += np.abs(accuracy_in_bin - avg_confidence_in_bin) * prop_in_bin

    return ece

# ==========================================
# 2. THE THESIS AUDIT SCRIPT
# ==========================================
def run_robustness_audit():
    print(">>> INITIATING THESIS MATHEMATICAL AUDIT <<<")

    # --- LOAD CACHED DATA ---
    print("Loading cached matrices and metadata...")
    vis_embeds = np.load("cached_vision_embeddings_1744.npy")
    txt_embeds = np.load("cached_text_embeddings_custom.npy")
    logit_scale = np.load("flair_logit_scale.npy").item()

    vis_meta = pd.read_csv("cached_vision_metadata_1744.csv")
    txt_meta = pd.read_csv("cached_text_metadata_custom.csv")

    # --- NORMALIZATION ---
    # To compute cosine similarity, both vectors must be L2 normalized
    # so they rest on the same unit hypersphere.
    vis_embeds = vis_embeds / np.linalg.norm(vis_embeds, axis=1, keepdims=True)
    txt_embeds = txt_embeds / np.linalg.norm(txt_embeds, axis=1, keepdims=True)

    # Map the JSON shorthand grades to the numerical values used in the vision CSV
    grade_mapping = {'noDR': 0, 'mildDR': 1, 'modDR': 2, 'sevDR': 3, 'prolDR': 4}
    txt_meta['numerical_grade'] = txt_meta['Grade'].map(grade_mapping)

    true_labels = vis_meta['adjudicated_dr_grade'].values

    # Get the unique prompt families (T_expert, T_layman, T_label, T_exclusion)
    families = txt_meta['Family'].unique()

    results = []

    # --- EXECUTE THE EVALUATION ---
    for family in families:
        print(f"\nEvaluating Prompt Family: {family}...")

        family_accuracies = []
        family_confidences = []

        # Test each of the 10 variations independently
        for var_num in range(1, 11):
            # Isolate the 5 text prompts for this specific variation and family
            var_mask = (txt_meta['Family'] == family) & (txt_meta['Variation_Number'] == var_num)
            var_meta = txt_meta[var_mask].sort_values('numerical_grade')

            if len(var_meta) != 5:
                continue # Skip if data is malformed

            # Extract the 5 corresponding text vectors
            var_txt_embeds = txt_embeds[var_meta.index.values]

            # -----------------------------------------------------
            # THE CRITICAL MATHEMATICAL STEP
            # 1. Dot Product (Cosine Similarity because vectors are normalized)
            # 2. Multiply by the learned Temperature (tau)
            # -----------------------------------------------------
            raw_similarities = np.dot(vis_embeds, var_txt_embeds.T)
            scaled_logits = raw_similarities * logit_scale

            # Apply Softmax to convert raw math into confidence percentages (probabilities)
            exp_logits = np.exp(scaled_logits - np.max(scaled_logits, axis=1, keepdims=True))
            probabilities = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

            # The model's predicted DR grade (0 through 4) is the column with the highest probability
            predictions = np.argmax(probabilities, axis=1)
            # The confidence is that exact highest probability value
            confidences = np.max(probabilities, axis=1)

            # Calculate binary accuracy (1 if correct, 0 if wrong) for ECE math
            accuracies = (predictions == true_labels).astype(float)

            # Pool the data
            family_accuracies.extend(accuracies)
            family_confidences.extend(confidences)

        # Compute final metrics for the entire prompt family
        if family_accuracies:
            overall_acc = np.mean(family_accuracies)
            overall_ece = calculate_ece(np.array(family_confidences), np.array(family_accuracies), num_bins=10)

            results.append({
                'Prompt Family': family,
                'Accuracy': f"{overall_acc * 100:.2f}%",
                'ECE': f"{overall_ece * 100:.2f}%",
                'Avg Confidence': f"{np.mean(family_confidences) * 100:.2f}%"
            })
            print(f"  -> Accuracy: {overall_acc * 100:.2f}% | ECE: {overall_ece * 100:.2f}% | Confidence: {np.mean(family_confidences) * 100:.2f}%")

    # --- SAVE RESULTS ---
    results_df = pd.DataFrame(results)
    results_df.to_csv("thesis_audit_results_1744.csv", index=False)
    print("\n>>> AUDIT COMPLETE. RESULTS SAVED TO 'thesis_audit_results_1744.csv' <<<")

if __name__ == "__main__":
    run_robustness_audit()

In [ ]:
import numpy as np
import pandas as pd

def compute_ece(confidences, accuracies, num_bins=10):
    bin_boundaries = np.linspace(0, 1, num_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    ece = 0.0
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.mean()
        if prop_in_bin > 0:
            accuracy_in_bin = accuracies[in_bin].mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    return ece

def regenerate_tables():
    # 1. Load Data
    v_embeds = np.load("cached_vision_embeddings_1744.npy")
    t_embeds = np.load("cached_text_embeddings_custom.npy")
    tau = np.load("flair_logit_scale.npy").item()

    v_meta = pd.read_csv("cached_vision_metadata_1744.csv")
    t_meta = pd.read_csv("cached_text_metadata_custom.csv")

    true_labels = v_meta['adjudicated_dr_grade'].values
    dr_mapping = {0.0:0, 1.0:1, 2.0:2, 3.0:3, 4.0:4}
    y_true = np.array([dr_mapping[val] for val in true_labels])

    families = t_meta['Family'].unique() # Adjust if using a custom grouping

    table_4_1_data = []
    table_4_2_data = []

    # 2. Iterative Evaluation
    for family in families:
        family_df = t_meta[t_meta['Family'] == family]
        variations = family_df['Variation_Number'].unique()

        var_accuracies = []
        var_eces = []

        all_preds = []
        all_confs = []
        all_accs_binary = []

        for var in variations:
            var_indices = family_df[family_df['Variation_Number'] == var].index.tolist()
            T_v = t_embeds[var_indices]

            # Dot Product and Scaling
            logits = (v_embeds @ T_v.T) * tau

            # Softmax
            exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
            probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

            preds = np.argmax(probs, axis=1)
            confs = np.max(probs, axis=1)
            accs_binary = (preds == y_true).astype(float)

            # Per-Variation Metrics (Table 4.2)
            var_acc = np.mean(accs_binary)
            var_ece = compute_ece(confs, accs_binary)
            var_accuracies.append(var_acc)
            var_eces.append(var_ece)

            # Pooling (Table 4.1)
            all_confs.extend(confs)
            all_accs_binary.extend(accs_binary)

        all_confs = np.array(all_confs)
        all_accs_binary = np.array(all_accs_binary)

        # Global Metrics
        global_acc = np.mean(all_accs_binary)
        global_ece = compute_ece(all_confs, all_accs_binary)
        global_conf = np.mean(all_confs)

        table_4_1_data.append((family, global_acc*100, global_ece*100, global_conf*100))
        table_4_2_data.append((family, np.mean(var_accuracies), np.std(var_accuracies), np.mean(var_eces), np.std(var_eces)))

    # 3. LaTeX Compilation
    print("% === UPDATED TABLE 4.1 ===")
    print("\\begin{table}[htbp]\n    \\centering")
    print("    \\caption{Quantitative Performance and Calibration Summary of the Frozen FLAIR Architecture Across Semantic Prompt Taxonomies.}")
    print("    \\label{tab:results_summary}\n    \\begin{tabular}{@{}lccc@{}}\n        \\toprule")
    print("        \\textbf{Prompt Family} & \\textbf{Accuracy (\\%)} & \\textbf{ECE (\\%)} & \\textbf{Avg. Confidence (\\%)} \\\\n        \\midrule")
    for row in table_4_1_data:
        print(f"        {row[0]} & {row[1]:.2f} & {row[2]:.2f} & {row[3]:.2f} \\\\n")
    print("        \\bottomrule\n    \\end{tabular}\n\\end{table}\n")

    print("% === UPDATED TABLE 4.2 ===")
    print("\\begin{table}[htbp]\n    \\centering")
    print("    \\caption{Prompt Sensitivity and Calibration Variance.}")
    print("    \\label{tab:variance_calibration}\n    \\begin{tabular}{@{}lcc@{}}\n        \\toprule")
    print("        \\textbf{Prompt Family} & \\textbf{Accuracy} & \\textbf{ECE} \\\\n        \\midrule")
    for row in table_4_2_data:
        print(f"        {row[0]} & ${row[1]:.4f} \\pm {row[2]:.4f}$ & ${row[3]:.4f} \\pm {row[4]:.4f}$ \\\\n")
    print("        \\bottomrule\n    \\end{tabular}\n\\end{table}")

if __name__ == "__main__":
    regenerate_tables()